# NumPy Tensor Tutorial

This workflow demonstrates how to transform raw, multi-rate EEG FIF files into a standardized, ML-ready 3D NumPy tensor using `lcmv_xtra` and `lcmv_stats`.

### **Workflow Overview**

1.  **Scan** complex directories for specific EEG conditions using flexible glob patterns.
2.  **Process** data in parallel via LCMV source estimation and CIMT atlas extraction.
3.  **Standardize** mixed sampling rates (e.g., 250/500 Hz) into a single uniform 3D tensor.
4.  **Extract** spectral features from continuous source-space data for Machine Learning.



In [ ]:
# =============================================================================
# 1. CONFIGURATION
# =============================================================================

import warnings
warnings.filterwarnings("ignore", message=".*does not conform to MNE naming conventions.*")

import lcmv_xtra as lx
from pathlib import Path

CLEAN_DIR = Path("/mnt/movement/users/jaizor/xtra/derivatives/eeg/rest/clean")
FS_DIR = Path("/mnt/movement/users/jaizor/xtra/derivatives/_fs")
OUTPUT_DIR = Path("./ml_data")
PROJECT_BASE = Path("/mnt/movement/users/jaizor/xtra")

In [2]:
# =============================================================================
# 2. EXTRACT EYES CLOSED ONLY
# =============================================================================

print(">>> Scanning for Eyes Closed (Central) files...")
# The 'c' in the filename corresponds to your Central/Eyes Closed condition
df_closed = lx.scan_eeg_paths(CLEAN_DIR, pattern="*_c_eeg_mkit_cleaned.fif")

if not df_closed.empty:
    print(f"Found {len(df_closed)} subjects. Starting parallel processing...")
    
    lx.assemble_tensor(
        data_index=df_closed,
        fs_dir=FS_DIR,
        output_dir=OUTPUT_DIR,
        task_name="eyes_closed", # This will save as study_eyes_closed.npz
        project_base=PROJECT_BASE,
        n_jobs=-1,               # Use all CPU cores
        target_sfreq=250.0       # <--- FORCE EVERYONE TO 250 Hz
    )
    
    print("\n✅ Eyes Closed tensor saved successfully!")
else:
    print("❌ No Eyes Closed files found. Check your CLEAN_DIR path.")


>>> Scanning for Eyes Closed (Central) files...
Found 12 subjects. Starting parallel processing...

✅ Eyes Closed tensor saved successfully!


In [ ]:
# =============================================================================
# 3. LOAD DATA
# =============================================================================

import numpy as np

# Load the saved tensor
master = np.load("./ml_data/study_eyes_closed.npz")

# Inspect what's inside
print(master.files)  # ['data', 'subject_ids', 'sfreq']

# Access the components
data = master['data']          # Shape: (12, 448, T) → (Subjects, ROIs, Time)
subjects = master['subject_ids']  # Array of subject labels
sfreq = master['sfreq']        # Sampling frequency (should now be 250.0)

print(f"Tensor shape: {data.shape}")
print(f"Subjects: {subjects}")
print(f"Sampling rate: {sfreq} Hz")
print(f"Duration: {data.shape[2] / sfreq:.2f} seconds")

['data', 'subject_ids', 'sfreq']
Tensor shape: (12, 448, 15000)
Subjects: ['sub-11' 'sub-03' 'sub-07' 'sub-05' 'sub-14' 'sub-02' 'sub-10' 'sub-12'
 'sub-09' 'sub-01' 'sub-06' 'sub-08']
Sampling rate: 250.0 Hz
Duration: 60.00 seconds


In [ ]:
# =============================================================================
# 4. SELECT A SINGLE ROI
# =============================================================================

import numpy as np
import pandas as pd
import lcmv_stats as ls

# 1. Load the Master Tensor
master = np.load("./ml_data/study_eyes_closed.npz")
data = master['data']       # (12, 448, 15000)
subjects = master['subject_ids']
sfreq = float(master['sfreq']) # 250.0

# 2. Extract a specific ROI for all subjects
roi_idx = ls.get_roi_index("R_4_ROI")
motor_signals = data[:, roi_idx, :] # Shape: (12, 15000)

# 3. Feature Extraction for Single Condition
all_features = []

print(">>> Extracting Spectral Features for Eyes Closed...")
for i, sid in enumerate(subjects):
    signal_1d = motor_signals[i, :]
    
    # Step A: Z-Score Normalization
    signal_z = ls.zscore_normalization(signal_1d)
    
    # Step B: Create Epochs (2s duration, 50% overlap)
    epochs = ls.create_epochs(
        signal_data=signal_z, 
        fs=sfreq, 
        epoch_duration=2.0, 
        overlap_frac=0.5
    )
    
    # Step C: Compute Features (Power in Delta, Theta, Alpha, Beta, Gamma)
    df_sub = ls.compute_epoch_features(epochs, fs=sfreq)
    
    # Add subject ID to track who is who
    df_sub['subject_id'] = sid
    
    all_features.append(df_sub)

# Combine all subjects into one big DataFrame
df_ml = pd.concat(all_features, ignore_index=True)

print(f"\n✅ Total epochs created: {len(df_ml)}")
print(f"   Subjects: {df_ml['subject_id'].nunique()}")
print(f"   Features per epoch: {len(df_ml.columns) - 2}") # minus epoch index and subject_id


>>> Extracting Spectral Features for Eyes Closed...

✅ Total epochs created: 708
   Subjects: 12
   Features per epoch: 6


In [8]:
df_ml.head()

,epoch,delta,theta,alpha,beta,low_gamma,high_gamma,subject_id
0,0,0.334421,0.047386,0.096885,0.215663,0.136321,0.148901,sub-11
1,1,0.198918,0.075792,0.051205,0.244110,0.158497,0.195561,sub-11
2,2,0.215801,0.107970,0.063724,0.241526,0.213395,0.214387,sub-11
3,3,0.051548,0.013756,0.050986,0.152298,0.153656,0.201566,sub-11
4,4,0.294089,0.101500,0.044650,0.239402,0.168158,0.245381,sub-11
